<a href="https://colab.research.google.com/github/Kate6097/train/blob/important-functions/%D0%A6%D0%B8%D0%BA%D0%BB%20%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
DIRECTION_LOSS_INTERVAL = 50 # Как часто пересчитываем, какие модули оставлять активными
NUM_TRAINABLE_LAYERS = 27    # Сколько модулей оставляем активными

BATCH_SIZE = 10
optimizer_g = optim.AdamW(
    generator_trainable.parameters(),
    lr=1e-3*5,
    betas=(0.9, 0.999)
)



num_steps = 300  # количество шагов оптимизации

scheduler = CosineAnnealingLR(optimizer_g, T_max=num_steps)
batch = generate_latent_batch(BATCH_SIZE, device, generator_frozen.style)
NUM_VIZ_SAMPLES = 5
fixed_latent_batch_for_viz = generate_latent_batch(NUM_VIZ_SAMPLES, device, generator_frozen.style)
torch.backends.cudnn.benchmark = True
torch.set_grad_enabled(True)
# Не забудьте сохранять лоссы для дальнейшей визуализации!
losses = {'clip_selection': [], 'training': []}

for step in range(num_steps):
    torch.cuda.empty_cache()

    if step % DIRECTION_LOSS_INTERVAL == 0:
        print(f"\n--- Шаг {step}: Пересчёт важности слоёв ---")
        latent = batch[np.random.randint(0, BATCH_SIZE)].unsqueeze(0)
        for param in generator_trainable.parameters():
            param.requires_grad_(True)
        generator_trainable.zero_grad()
        img_for_analysis, _ = generator_trainable([latent], input_is_latent=True, randomize_noise=True)

        selection_loss, active_layers_names = analyze_and_set_layers(
            generator_trainable,
            img_for_analysis,
            prompt2,
            clip_loss,
            num_trainable=NUM_TRAINABLE_LAYERS
        )
        losses['clip_selection'].append(selection_loss)
        print(f"Активные слои: {active_layers_names[:5]}...") # Выведем несколько для проверки

        current_lr = optimizer_g.param_groups[0]['lr']
        #насколько я поняла, после перезаморозки лучше заново инициализировать оптимайзер
        optimizer_g = optim.Adam(
            filter(lambda p: p.requires_grad, generator_trainable.parameters()),
            lr=current_lr, # Используем текущий LR
            betas=(0.9, 0.999)
        )
        # T_max для scheduler также должен учитывать оставшиеся шаги
        scheduler = CosineAnnealingLR(optimizer_g, T_max=num_steps - step)

    optimizer_g.zero_grad()


    lambda_directional = 10.0 # Вес для directional loss, увеличила, т к модель лениво добавляла признаки
    lambda_target = 0.0 #часть лосса с clip вообще по статье не нужна, но я экспериментировала и сейчас эту часть лосса просто занулила
    total_loss = 0
    for i in range(BATCH_SIZE):
      latent_i = batch[i:i+1]

      img_gen_i, _ = generator_trainable([latent_i], input_is_latent=True, randomize_noise=True)

      with torch.no_grad():
          img_gen_frozen_i, _ = generator_frozen([latent_i], input_is_latent=True, randomize_noise=True)

      current_loss = direct_loss(text_in, text_out, img_gen_i, img_gen_frozen_i, beta=0.1)
      clip_image = normalize(resize(img_gen_i))
      current_clip_loss = clip_loss(clip_image, text_out)

      total_loss += (lambda_directional * current_loss) + (lambda_target * current_clip_loss)

      # Явное удаление тензоров, на всякий случай, так как при моем размере батча могут возникнуть проблемы с памятью
      del img_gen_i, img_gen_frozen_i, clip_image, current_loss, current_clip_loss
      torch.cuda.empty_cache() # Очистка кэша после каждого элемента батча
    training_loss = total_loss / BATCH_SIZE
    training_loss.backward()
    optimizer_g.step()

    scheduler.step()
    losses['training'].append(training_loss.item())

    # Визуализация и вывод потерь
    if step % 50 == 0:
        print(f"Шаг {step}: Тренировочный лосс = {training_loss.item():.4f}")

        # Генерация изображений с ФИКСИРОВАННЫМИ латентными векторами для визуализации
        with torch.no_grad():
            img_frozen_batch, _ = generator_frozen(
                [fixed_latent_batch_for_viz],
                input_is_latent=True,
                randomize_noise=False
            )

            generator_trainable.eval()
            img_trainable_batch, _ = generator_trainable(
                [fixed_latent_batch_for_viz],
                input_is_latent=True,
                randomize_noise=False
            )
            generator_trainable.train()


        fig, axs = plt.subplots(2, NUM_VIZ_SAMPLES, figsize=(2 * NUM_VIZ_SAMPLES, 4))

        def prepare_img(img_tensor):
            img = img_tensor.cpu().detach().clip(-1, 1).permute(1, 2, 0)
            return (img + 1) / 2

        for i in range(NUM_VIZ_SAMPLES):
            axs[0, i].imshow(prepare_img(img_trainable_batch[i]))
            axs[0, i].set_title(f"Обучается {i+1}")
            axs[0, i].axis('off')

        for i in range(NUM_VIZ_SAMPLES):
            axs[1, i].imshow(prepare_img(img_frozen_batch[i]))
            axs[1, i].set_title(f"Замороженное {i+1}")
            axs[1, i].axis('off')

        plt.tight_layout()
        plt.show()

        #также стараемся беречь память при каждом удобном случае
        del img_frozen_batch, img_trainable_batch, fig, axs
        torch.cuda.empty_cache()

